# 10 · Storage & versioning — lakeFS (git-for-data)

**lakeFS is git, for data.** It layers a version-control model over the object store
(MinIO, here) that already holds the lake's bytes — so the same moves you know from
git apply to terabytes of Parquet, Arrow, Avro and Lance:

| git | lakeFS |
|-----|--------|
| `branch` | **zero-copy branch** — a new isolated namespace over the *same* objects, created instantly and for free (no bytes are duplicated) |
| `commit` | an **immutable, addressable snapshot** of the whole repo at a point in time |
| `diff` | what objects were **added / changed / removed** between two refs |
| `merge` | fold one branch's commits into another |
| `log` | the **commit history** of a branch |
| `checkout <sha>` | **time-travel** — read the lake exactly as it was at any commit |

### Where it sits in the weyland data mesh

lakeFS is the **storage & versioning layer**. Raw and curated datasets land as objects
on branches of a lakeFS repo (backed by MinIO); everything above — Trino, dbt marts,
the format-comparison notebooks in this folder — reads them through lakeFS's **S3
gateway** at a chosen ref. That gateway is why the seed notebook `datasets_lake.ipynb`
can point plain `s3fs`/`polars` at `s3://music/main/...` and get versioned data back.

This notebook uses the existing **`music`** repository and demonstrates the full
git-for-data loop **without touching any real branch** — every write happens on
throwaway scratch branches that are deleted again at the end.

> **Two tools, two jobs.** We use the high-level **`lakefs` SDK** for *versioning*
> (branch / commit / diff / merge / log / delete) and **`s3fs`** for moving *object
> bytes* on a branch — exactly the split the seed notebook uses for reads.


## Setup

The `lakefs` SDK is **not** in the singleuser base image, so we install it here.
`s3fs`, `polars` and `pyarrow` already ship in the image.

In [1]:
%pip install -q lakefs

Note: you may need to restart the kernel to use updated packages.


## 2 · Connect

Connection is **env-driven** — the singleuser pod injects `LAKEFS_ENDPOINT` and the
credential pair. The committed default endpoint is the in-cluster service URL; a
validation run overrides `LAKEFS_ENDPOINT` (e.g. to a `kubectl port-forward`) without
editing the notebook.

We build:
- a **`lakefs.Client`** (host + key/secret) for all versioning calls, and
- an **`s3fs.S3FileSystem`** pointed at the same endpoint for object bytes — mirroring
  `datasets_lake.ipynb` (path-style, plain http, region `us-east-1`).

In [2]:
import os, io, time
import lakefs
from lakefs.client import Client
import s3fs
import polars as pl

LAKEFS_ENDPOINT = os.environ.get('LAKEFS_ENDPOINT', 'http://lakefs.data-mesh.svc.cluster.local:8000')
KEY, SECRET = os.environ['LAKEFS_ACCESS_KEY_ID'], os.environ['LAKEFS_SECRET_ACCESS_KEY']

REPO = 'music'
SRC_BRANCH = 'main'                       # never written to — read-only reference
DEMO_BRANCH = 'notebook-demo'             # scratch — created + deleted by this notebook
TARGET_BRANCH = 'notebook-demo-target'    # scratch merge target — also cleaned up
PLAYGROUND = 'notebook-demo/_nb_playground'   # clearly-scratch object prefix

# versioning client (lakefs SDK)
client = Client(host=LAKEFS_ENDPOINT, username=KEY, password=SECRET)

# byte I/O on branches (s3fs through the lakeFS S3 gateway) — same options as datasets_lake.ipynb
fs = s3fs.S3FileSystem(key=KEY, secret=SECRET, use_ssl=False,
                       client_kwargs={'endpoint_url': LAKEFS_ENDPOINT, 'region_name': 'us-east-1'})

print('lakeFS endpoint :', LAKEFS_ENDPOINT)
print('lakeFS server   :', client.version)   # server version, proves the connection
print('repositories    :', [r.id for r in lakefs.repositories(client=client)])

lakeFS endpoint : http://localhost:8899
lakeFS server   : 1.67.0
repositories    : ['health', 'music']


### Idempotent start — remove any leftover scratch branches

If a previous run crashed before cleanup, the scratch branches might still exist.
Delete them up front (ignoring *not-found*) so this notebook always starts from a
clean slate. **Only the two `notebook-demo*` scratch branches are ever touched.**

In [3]:
def drop_scratch_branch(name):
    '''Delete a scratch branch if it exists; ignore not-found. Refuses anything non-scratch.'''
    assert name.startswith('notebook-demo'), f'refusing to touch non-scratch branch {name!r}'
    try:
        lakefs.Repository(REPO, client=client).branch(name).delete()
        return f'deleted leftover {name}'
    except lakefs.exceptions.NotFoundException:
        return f'{name} absent (clean)'

repo = lakefs.Repository(REPO, client=client)
for _b in (DEMO_BRANCH, TARGET_BRANCH):
    print(drop_scratch_branch(_b))

notebook-demo absent (clean)
notebook-demo-target absent (clean)


## 3 · Explore the `music` repository (read-only)

Before we branch, let's look at what's already there — the branches, `main`'s commit
history, and a sample of the objects living on `main`. This is all read-only; `main`
is never modified.

In [4]:
print('branches in', REPO, ':', [b.id for b in repo.branches()])

branches in music : ['main']


**`main`'s commit history** (`Reference.log`) — the immutable snapshots, newest first:

In [5]:
main = repo.ref(SRC_BRANCH)

rows = []
for cm in main.log(max_amount=8):
    rows.append({
        'commit':    cm.id[:12],
        'committer': cm.committer,
        'when':      time.strftime('%Y-%m-%d %H:%M', time.localtime(cm.creation_date)),
        'message':   (cm.message or '')[:60],
    })
pl.DataFrame(rows)

commit,committer,when,message
str,str,str,str
"""4b7b0b7c93e9""","""admin""","""2026-08-31 02:35""","""mart_spotify_audio_export (dbt…"
"""44ef097e658a""","""admin""","""2026-08-25 02:35""","""mart_spotify_audio_export (dbt…"
"""be3d986492ce""","""admin""","""2026-08-24 02:36""","""mart_spotify_audio_export (dbt…"
"""f8c69b09e68e""","""admin""","""2026-08-16 13:49""","""mart_spotify_audio_export (dbt…"
"""4cccb9dc11cc""","""admin""","""2026-08-10 02:36""","""mart_spotify_audio_export (dbt…"
"""52079fb0c861""","""admin""","""2026-08-08 02:36""","""genre_feast_training_set (Feas…"
"""bed3b6d8614c""","""admin""","""2026-08-07 02:36""","""genre_feast_training_set (Feas…"
"""99acedd35b08""","""admin""","""2026-08-03 02:37""","""mart_spotify_audio_export (dbt…"


**A listing of objects on `main`** (`Reference.objects`) — a peek at the lake's contents:

In [6]:
main_objs = []
for o in main.objects(max_amount=12):
    # objects() can yield CommonPrefix entries too; keep only real objects
    if getattr(o, 'path', None) is not None and getattr(o, 'size_bytes', None) is not None:
        main_objs.append({'path': o.path, 'size_bytes': o.size_bytes})
pl.DataFrame(main_objs)

path,size_bytes
str,i64
"""arrow/audioset/test.arrow""",703554
"""arrow/audioset/train.arrow""",745290
"""arrow/fma_echonest/1782666876.…",59362
"""arrow/fma_echonest/1782666876.…",24002394
"""arrow/fma_echonest/1782678943.…",24009498
…,…
"""arrow/fma_genres/1782666876.62…",11122
"""arrow/fma_genres/1782678943.68…",11122
"""arrow/fma_genres/fma_genres.ar…",7746


## 4 · Zero-copy branch off `main`

Creating `notebook-demo` off `main` is **instant and free** — lakeFS copies *no bytes*,
it just points a new namespace at `main`'s current commit. Proof: the new branch's
HEAD commit id **equals** `main`'s HEAD, and listing objects on it returns exactly what
`main` has.

In [7]:
demo = repo.branch(DEMO_BRANCH).create(source_reference=SRC_BRANCH, exist_ok=True)

main_head = main.get_commit().id
demo_head = demo.head.id
print('main   HEAD:', main_head[:12])
print('demo   HEAD:', demo_head[:12])
print('same commit — zero-copy branch:', main_head == demo_head)

# objects visible on the fresh branch == objects on main (shared, not duplicated)
demo_count = sum(1 for o in demo.objects(max_amount=12) if getattr(o, 'path', None) is not None)
print(f'objects visible on {DEMO_BRANCH} (first page): {demo_count} — the same data main sees')

main   HEAD: 4b7b0b7c93e9
demo   HEAD: 4b7b0b7c93e9
same commit — zero-copy branch: True
objects visible on notebook-demo (first page): 12 — the same data main sees


## 5 · Write an object, then commit it

Now we write a small Parquet file to the **scratch prefix** on the scratch branch via
`s3fs` — path shape `s3://<repo>/<branch>/<key>`, exactly the gateway pattern from
`datasets_lake.ipynb`. The write leaves an **uncommitted change**; `Branch.commit`
turns it into an immutable snapshot and returns the new commit reference.

`main` sees none of this — the write is isolated on `notebook-demo`.

In [8]:
# a tiny demo dataset
demo_df = pl.DataFrame({
    'track_id':   [1, 2, 3, 4, 5],
    'title':      ['aria', 'nocturne', 'etude', 'prelude', 'fugue'],
    'plays':      [120, 340, 90, 510, 275],
})

object_key = f'{PLAYGROUND}/plays.parquet'            # notebook-demo/_nb_playground/plays.parquet
s3_uri = f'{REPO}/{DEMO_BRANCH}/{object_key}'         # write target on the scratch branch

buf = io.BytesIO()
demo_df.write_parquet(buf)
with fs.open(s3_uri, 'wb') as f:
    f.write(buf.getvalue())
print('wrote', f's3://{s3_uri}', f'({buf.tell()} bytes)')

# the write is uncommitted until we commit it
uncommitted = [c.path for c in demo.uncommitted()]
print('uncommitted on', DEMO_BRANCH, ':', uncommitted)

wrote s3://music/notebook-demo/notebook-demo/_nb_playground/plays.parquet (1278 bytes)
uncommitted on notebook-demo : ['notebook-demo/_nb_playground/plays.parquet']


In [9]:
commit_ref = demo.commit(
    message='notebook-demo: add plays.parquet to _nb_playground',
    metadata={'source': '10_storage_lakefs.ipynb', 'kind': 'demo'},
)
DEMO_COMMIT = commit_ref.get_commit().id
print('committed — new commit id:', DEMO_COMMIT[:12])
print('nothing uncommitted now  :', [c.path for c in demo.uncommitted()] or 'clean')

committed — new commit id: 4e6fb40c0578
nothing uncommitted now  : clean


## 6 · Diff `main` vs `notebook-demo`

`Reference.diff` shows what changed between two refs. Our one added object appears as
an `added` change — and `main` itself is untouched.

In [10]:
changes = list(main.diff(other_ref=demo))
pl.DataFrame([{'type': c.type, 'path': c.path} for c in changes])

type,path
str,str
"""added""","""notebook-demo/_nb_playground/p…"


## 7 · Merge — safely, into a scratch target (never `main`)

To demonstrate merge without any risk to real data, we create a **second** scratch
branch off `main` and merge `notebook-demo` **into that target** — never into `main`.
After the merge, the demo object is present on the target branch.

In [11]:
target = repo.branch(TARGET_BRANCH).create(source_reference=SRC_BRANCH, exist_ok=True)
print('created scratch merge target:', TARGET_BRANCH, '@', target.head.id[:12])

merge_commit = demo.merge_into(target)          # returns the merge commit id
print('merged', DEMO_BRANCH, '->', TARGET_BRANCH, '=> merge commit', str(merge_commit)[:12])

# the object is now on the target branch
on_target = [o.path for o in target.objects(prefix=PLAYGROUND + '/')
             if getattr(o, 'path', None) is not None]
print('objects under scratch prefix on target:', on_target)

created scratch merge target: notebook-demo-target @ 4b7b0b7c93e9


merged notebook-demo -> notebook-demo-target => merge commit 788194672d0a
objects under scratch prefix on target: ['notebook-demo/_nb_playground/plays.parquet']


## 8 · History & time-travel

`Branch.log` gives the branch's commit history — our demo commit now sits on top of the
inherited `main` history. And because every commit is immutable and addressable, we can
**read the object as of a specific commit** (time-travel) by using the commit id as the
ref in the S3-gateway path: `s3://<repo>/<commit-id>/<key>`. This proves you can read
the lake at *any* point in its history, not just the branch tip.

In [12]:
hist = []
for cm in demo.log(max_amount=6):
    hist.append({
        'commit':  cm.id[:12],
        'when':    time.strftime('%Y-%m-%d %H:%M', time.localtime(cm.creation_date)),
        'message': (cm.message or '')[:60],
    })
pl.DataFrame(hist)

commit,when,message
str,str,str
"""4e6fb40c0578""","""2026-09-01 13:40""","""notebook-demo: add plays.parqu…"
"""4b7b0b7c93e9""","""2026-08-31 02:35""","""mart_spotify_audio_export (dbt…"
"""44ef097e658a""","""2026-08-25 02:35""","""mart_spotify_audio_export (dbt…"
"""be3d986492ce""","""2026-08-24 02:36""","""mart_spotify_audio_export (dbt…"
"""f8c69b09e68e""","""2026-08-16 13:49""","""mart_spotify_audio_export (dbt…"
"""4cccb9dc11cc""","""2026-08-10 02:36""","""mart_spotify_audio_export (dbt…"


In [13]:
# time-travel: read the demo object AT the exact commit that introduced it
fs.invalidate_cache()
timetravel_uri = f'{REPO}/{DEMO_COMMIT}/{object_key}'   # note: commit id, not a branch name
with fs.open(timetravel_uri, 'rb') as f:
    at_commit = pl.read_parquet(io.BytesIO(f.read()))

print(f'read s3://{REPO}/{DEMO_COMMIT[:12]}.../{object_key}')
print(f'rows={at_commit.height} cols={at_commit.width} — recovered from an immutable commit')
at_commit

read s3://music/4e6fb40c0578.../notebook-demo/_nb_playground/plays.parquet
rows=5 cols=3 — recovered from an immutable commit


track_id,title,plays
i64,str,i64
1,"""aria""",120
2,"""nocturne""",340
3,"""etude""",90
4,"""prelude""",510
5,"""fugue""",275


## 9 · Cleanup — leave lakeFS exactly as we found it

Delete **both** scratch branches and assert they no longer appear in the branch list.
This runs even if a step above failed, so the notebook never leaves residue behind.
Deleting the branches discards every commit and object we created; `main` and every
real branch are untouched.

In [14]:
cleanup_report = {}
try:
    pass  # (all demo work already done above)
finally:
    for _b in (DEMO_BRANCH, TARGET_BRANCH):
        try:
            repo.branch(_b).delete()
            cleanup_report[_b] = 'deleted'
        except lakefs.exceptions.NotFoundException:
            cleanup_report[_b] = 'already absent'

remaining = [b.id for b in repo.branches() if b.id.startswith('notebook-demo')]
print('cleanup:', cleanup_report)
print('scratch branches remaining:', remaining)
assert remaining == [], f'scratch branches left behind: {remaining}'
print('OK — lakeFS left exactly as found (only', SRC_BRANCH, 'and real branches remain)')

cleanup: {'notebook-demo': 'deleted', 'notebook-demo-target': 'deleted'}
scratch branches remaining: []
OK — lakeFS left exactly as found (only main and real branches remain)


## 10 · When to reach for lakeFS — and how it pairs with Iceberg/Nessie

**Use lakeFS when you need git semantics over *objects*:**
- **Reproducibility** — pin a training run, a dbt build, or a report to an exact commit
  id and re-read the lake as it was then (the time-travel we just did).
- **Safe experimentation** — branch the whole lake for free, rewrite datasets on the
  branch, throw the branch away if the experiment fails. Zero blast radius on `main`.
- **Data CI/CD** — ingest onto a branch, run quality/expectation checks against it, and
  only **merge to `main`** when the checks pass — an atomic, all-or-nothing promotion.

**lakeFS vs Nessie/Iceberg** — they version at different layers and compose:
- **lakeFS** versions **objects** (the files themselves — Parquet, Arrow, Lance, images,
  anything) across the *whole* repo. Format-agnostic, one commit spans every dataset.
- **Nessie + Iceberg** version **tables** — Iceberg tracks table snapshots via manifests,
  and Nessie gives those a git-like branch/merge catalog. Table-aware (schema evolution,
  partition specs, row-level ops), scoped to Iceberg tables.

In the mesh they stack: lakeFS underneath for whole-lake object versioning and CI/CD,
Iceberg tables (catalogued by Nessie) on top for transactional, schema-evolving analytics.

➡️ **Next:** the Nessie/Iceberg notebook picks up here — same git-for-data idea, one layer up.
